# Fabric Graph — data preparation for 2 security graphs
Builds the **node and edge Delta tables** that Fabric Graph models read, from the Sentinel tables mirrored into Fabric.

| Graph | Tables written (prefix) | Window |
|---|---|---|
| **1. Identity Attack Graph** | `ia_*` — users, IPs, countries, apps, sign-in / attack edges | 7 days |
| **2. Incident Entity Graph** | `ie_*` — incidents, alerts, users, hosts, IPs and their links | 30 days |

Tables are written **by OneLake path** to the **SecAnalytics** Lakehouse (schema `dbo`), so no default Lakehouse is required. Check `WORKSPACE_ID` / `LAKEHOUSE_ID` in the configuration cell.
Schedule this notebook daily from a Data pipeline, then refresh the two graph models.

In [ ]:
# ── Configuration ───────────────────────────────────────────────
# Query URI of the SOC_Analytics Eventhouse (KQL database details pane -> Query URI)
KUSTO_URI   = 'https://trd-2gtxb2yrad9vq2b4qt.z3.kusto.fabric.microsoft.com'
SENTINEL_DB = 'MirrorSentinel'     # follower database with all Sentinel tables (external_table)
SOC_DB      = 'SOC_Analytics'      # database with EntraBruteForceFindings / DailyIncidentRiskFindings
IA_DAYS, IE_DAYS = 7, 30

# Target Lakehouse (SecAnalytics, schema-enabled) - written by path, so no default Lakehouse is needed
WORKSPACE_ID = '359c56b1-2d08-4c04-943e-ebbf31796b66'   # Sentinel_Datalake workspace (URL after /groups/)
LAKEHOUSE_ID = '8e4a9960-8dfb-4e3d-a259-345583caa1d8'   # SecAnalytics (URL after /lakehouses/)
TABLES_PATH  = f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/dbo'

from pyspark.sql import functions as F
try:
    _nbu = notebookutils
except NameError:
    from notebookutils import mssparkutils as _nbu

def kql(query, db=SENTINEL_DB):
    token = _nbu.credentials.getToken(KUSTO_URI)
    return (spark.read.format('com.microsoft.kusto.spark.synapse.datasource')
            .option('accessToken', token).option('kustoCluster', KUSTO_URI)
            .option('kustoDatabase', db).option('kustoQuery', query)
            .option('readMode', 'ForceSingleMode')   # avoid transient SAS storage that breaks OneLake writes
            .load())

def save(df, name):
    # Requires SecAnalytics as the notebook's default Lakehouse
    df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(f'dbo.{name}')
    print(f'{name}: {df.count()} rows')

def try_kql(query, db=SENTINEL_DB, label=''):
    try:
        return kql(query, db)
    except Exception as e:
        print(f'WARNING {label}: {str(e)[:200]}')
        return None

print(kql("external_table('SigninLogs') | take 1 | count").collect())

## Graph 1 — Identity Attack Graph
**Nodes:** `User`, `IPAddress`, `Country`, `App` · **Edges:** `signsInFrom` (User→IPAddress), `usesApp` (User→App), `attacked` (IPAddress→User), `locatedIn` (IPAddress→Country)

In [ ]:
SIGNINS = f'''union external_table('SigninLogs'), external_table('AADNonInteractiveUserSignInLogs')
| where TimeGenerated > ago({IA_DAYS}d)
| extend UserId = tolower(UserPrincipalName), R = tostring(ResultType)
| where isnotempty(UserId) and isnotempty(IPAddress)'''

# Edge: User -> IPAddress
signin_edges = kql(SIGNINS + '''
| summarize SignIns = count(), Failures = countif(R !in ("0","50140","50199")), Successes = countif(R == "0"),
            FirstSeen = min(TimeGenerated), LastSeen = max(TimeGenerated), Location = take_any(tostring(Location))
    by UserId, IP = tostring(IPAddress)
| extend G = geo_info_from_ip_address(IP)
| extend Country = iff(isempty(Location), tostring(G.country), Location), City = tostring(G.city),
         Latitude = todouble(G.latitude), Longitude = todouble(G.longitude)
| project UserId, IP, SignIns, Failures, Successes, FirstSeen, LastSeen, Country, City, Latitude, Longitude''').cache()
save(signin_edges.select('UserId','IP','SignIns','Failures','Successes','FirstSeen','LastSeen'), 'ia_edge_signsinfrom')

# Edge: User -> App
app_edges = kql(SIGNINS + '''
| where isnotempty(AppDisplayName)
| summarize SignIns = count(), Failures = countif(R !in ("0","50140","50199")), LastSeen = max(TimeGenerated)
    by UserId, App = tostring(AppDisplayName)''')
save(app_edges, 'ia_edge_usesapp')
save(app_edges.select('App').distinct(), 'ia_node_app')

# Edge: attacker IP -> targeted User (from the hourly brute-force pipeline)
attack = try_kql(f'''EntraBruteForceFindings | where TimeGenerated > ago({IA_DAYS}d)
| mv-expand UserId = split(TargetedUsers, ", ") to typeof(string)
| where isnotempty(UserId)
| extend Compromised = toint(indexof(CompromisedUsers, UserId) >= 0)
| summarize arg_max(TimeGenerated, Pattern, Severity), Windows = count(), Failures = sum(Failures), Compromised = max(Compromised)
    by IP = IPAddress, UserId''', SOC_DB, 'EntraBruteForceFindings')
if attack is None or attack.rdd.isEmpty():
    attack = spark.createDataFrame([], 'IP string, UserId string, TimeGenerated timestamp, Pattern string, Severity string, Windows long, Failures long, Compromised int')
save(attack, 'ia_edge_attacked')

In [ ]:
# Nodes: IPAddress (geo + attacker flag) and Country
ip_geo = (signin_edges.groupBy('IP').agg(F.first('Country', True).alias('Country'), F.first('City', True).alias('City'),
          F.first('Latitude', True).alias('Latitude'), F.first('Longitude', True).alias('Longitude'),
          F.countDistinct('UserId').alias('Users'), F.sum('Failures').alias('Failures'), F.sum('Successes').alias('Successes')))
_rank = (F.when(F.col('Severity') == 'Critical', 1).when(F.col('Severity') == 'High', 2)
          .when(F.col('Severity') == 'Medium', 3).otherwise(4))
att = (attack.groupBy('IP').agg(F.min(_rank).alias('SevRank'), F.countDistinct('UserId').alias('UsersAttacked'),
                               F.max('Compromised').alias('AnyCompromise'))
       .withColumn('MaxSeverity', F.when(F.col('SevRank') == 1, 'Critical').when(F.col('SevRank') == 2, 'High')
                   .when(F.col('SevRank') == 3, 'Medium').otherwise('Low')).drop('SevRank'))
ips = (ip_geo.select('IP').union(attack.select('IP')).distinct()
       .join(ip_geo, 'IP', 'left').join(att, 'IP', 'left')
       .withColumn('IsAttacker', F.col('UsersAttacked').isNotNull())
       .fillna({'Country': 'Unknown', 'Users': 0, 'Failures': 0, 'Successes': 0, 'UsersAttacked': 0, 'AnyCompromise': 0}))
save(ips, 'ia_node_ipaddress')
save(ips.select('IP', 'Country').distinct(), 'ia_edge_locatedin')
save(ips.select('Country').distinct(), 'ia_node_country')

# Nodes: User (every user referenced by any edge) + IdentityInfo enrichment
ident = try_kql('''external_table('IdentityInfo') | where TimeGenerated > ago(14d)
| summarize arg_max(TimeGenerated, *) by UserId = tolower(AccountUPN)
| project UserId, DisplayName = tostring(AccountDisplayName), Department = tostring(Department), JobTitle = tostring(JobTitle),
          IsPrivileged = tostring(AssignedRoles) has "Administrator"''', SENTINEL_DB, 'IdentityInfo')
users = (signin_edges.select('UserId').union(app_edges.select('UserId')).union(attack.select('UserId')).distinct())
users = users.join(signin_edges.groupBy('UserId').agg(F.sum('SignIns').alias('SignIns'), F.sum('Failures').alias('Failures'),
                                                      F.countDistinct('IP').alias('DistinctIPs'),
                                                      F.countDistinct('Country').alias('Countries')), 'UserId', 'left')
if ident is not None:
    users = users.join(ident, 'UserId', 'left')
else:
    users = users.withColumn('IsPrivileged', F.lit(False))
users = users.fillna({'SignIns': 0, 'Failures': 0, 'DistinctIPs': 0, 'Countries': 0, 'IsPrivileged': False})
save(users, 'ia_node_user')

## Graph 2 — Incident Entity Graph
**Nodes:** `Incident`, `Alert`, `User`, `Host`, `IPAddress` · **Edges:** `hasAlert` (Incident→Alert), `involvesUser`, `involvesHost`, `involvesIp` (Alert→entity)

In [ ]:
inc = kql(f'''external_table('SecurityIncident') | where TimeGenerated > ago({IE_DAYS}d)
| summarize arg_max(TimeGenerated, *) by IncidentNumber
| project IncidentId = tostring(IncidentNumber), Title = tostring(Title), Severity = tostring(Severity), Status = tostring(Status),
          Classification = tostring(Classification), CreatedTime = todatetime(CreatedTime),
          Owner = coalesce(tostring(todynamic(tostring(Owner)).assignedTo), "Unassigned"), AlertIds = tostring(AlertIds)''').cache()
risk = try_kql('''DailyIncidentRiskFindings | summarize arg_max(ReportDate, RiskScore, RiskTier) by IncidentId = tostring(IncidentNumber)
| project IncidentId, RiskScore, RiskTier''', SOC_DB, 'DailyIncidentRiskFindings')
inc_nodes = inc.drop('AlertIds')
if risk is not None:
    inc_nodes = inc_nodes.join(risk, 'IncidentId', 'left')
save(inc_nodes, 'ie_node_incident')

has_alert = (inc.select('IncidentId', F.from_json('AlertIds', 'array<string>').alias('A'))
             .select('IncidentId', F.explode_outer('A').alias('AlertId')).where(F.col('AlertId').isNotNull()).distinct())

alerts = kql(f'''external_table('SecurityAlert') | where TimeGenerated > ago({IE_DAYS}d)
| summarize arg_max(TimeGenerated, AlertName, AlertSeverity, Tactics, ProductName, Entities) by AlertId = tostring(SystemAlertId)''').cache()
has_alert = has_alert.join(alerts.select('AlertId'), 'AlertId', 'inner')          # keep only alerts we have as nodes
save(has_alert.select('IncidentId', 'AlertId'), 'ie_edge_hasalert')
save(alerts.select('AlertId', 'TimeGenerated', 'AlertName', F.col('AlertSeverity').alias('Severity'),
                   F.col('Tactics').cast('string').alias('Tactics'), 'ProductName'), 'ie_node_alert')

ent = kql(f'''external_table('SecurityAlert') | where TimeGenerated > ago({IE_DAYS}d)
| summarize arg_max(TimeGenerated, Entities) by AlertId = tostring(SystemAlertId)
| mv-expand E = todynamic(tostring(Entities))
| extend T = tolower(tostring(E.Type))
| extend EntityId = case(
    T == "account", tolower(iff(isnotempty(tostring(E.UPNSuffix)), strcat(tostring(E.Name), "@", tostring(E.UPNSuffix)), tostring(E.Name))),
    T == "mailbox", tolower(tostring(E.MailboxPrimaryAddress)),
    T == "host", tolower(tostring(E.HostName)),
    T == "ip", tostring(E.Address), "")
| extend Kind = case(T in ("account", "mailbox"), "User", T == "host", "Host", T == "ip", "IP", "")
| where isnotempty(EntityId) and isnotempty(Kind)
| distinct AlertId, Kind, EntityId''').cache()

for kind, node_tbl, edge_tbl, key in [('User', 'ie_node_user', 'ie_edge_involvesuser', 'UserId'),
                                      ('Host', 'ie_node_host', 'ie_edge_involveshost', 'HostId'),
                                      ('IP',   'ie_node_ipaddress', 'ie_edge_involvesip', 'IP')]:
    e = ent.where(F.col('Kind') == kind).select('AlertId', F.col('EntityId').alias(key))
    save(e, edge_tbl)
    n = e.groupBy(key).agg(F.countDistinct('AlertId').alias('Alerts'))
    if kind == 'IP':
        n = n.join(ips.select('IP', 'Country', 'IsAttacker'), 'IP', 'left')    # reuse geo from graph 1
    save(n, node_tbl)
print('done')